### **Importing Libraries**

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, BatchNormalization, Dropout, GlobalAveragePooling2D
import matplotlib.pyplot as plt
import cv2

### **Loading the Data and Analyzing the Structure**
We are using the EuroSAT dataset which contains 27,000 images of different terrain from the Earth surface

In [ ]:
import zipfile
import os

zip_path = "/content/EuroSAT.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content/EuroSAT")

In [ ]:
import os

data_dir = "/content/EuroSAT/2750"

classes = sorted(os.listdir(data_dir))

print("Number of classes:", len(classes))
print("\nClasses:")

for cls in classes:
    class_path = os.path.join(data_dir, cls)
    num_images = len(os.listdir(class_path))
    print(f"{cls:25s} {num_images}")

Number of classes: 10

Classes:
AnnualCrop                3000
Forest                    3000
HerbaceousVegetation      3000
Highway                   2500
Industrial                2500
Pasture                   2000
PermanentCrop             2500
Residential               3000
River                     2500
SeaLake                   3000


Analyzing the pixel structure

In [ ]:
from PIL import Image
import os

for cls in classes:
    class_path = os.path.join(data_dir, cls)
    first_image = os.listdir(class_path)[0]

    image_path = os.path.join(class_path, first_image)
    image = Image.open(image_path)

    print(f"{cls:25s} {image.size}  {image.mode}")

AnnualCrop                (64, 64)  RGB
Forest                    (64, 64)  RGB
HerbaceousVegetation      (64, 64)  RGB
Highway                   (64, 64)  RGB
Industrial                (64, 64)  RGB
Pasture                   (64, 64)  RGB
PermanentCrop             (64, 64)  RGB
Residential               (64, 64)  RGB
River                     (64, 64)  RGB
SeaLake                   (64, 64)  RGB


### **Initializing the Data Split**
We will be using a 80-20 Training-Validation Split and we will be converting the 64x64 Images into 150x150 as the VGG16 model is more apt to that structure. Batch size will be 64 images per batch.

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    directory='/content/EuroSAT/2750',
    labels='inferred',
    label_mode='int',
    validation_split=0.2,
    subset='training',
    seed=42,
    batch_size=64,
    image_size=(150,150)
)

validation_ds = keras.utils.image_dataset_from_directory(
    directory='/content/EuroSAT/2750',
    labels='inferred',
    label_mode='int',
    validation_split=0.2,
    subset='validation',
    seed=42,
    batch_size=64,
    image_size=(150,150)
)

Found 27000 files belonging to 10 classes.
Using 21600 files for training.
Found 27000 files belonging to 10 classes.
Using 5400 files for validation.


Verifying the Split

In [ ]:
class_names = train_ds.class_names

print("Classes:", class_names)

for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Pixel range:", images.numpy().min(), "to", images.numpy().max())
    print("First 10 labels:", labels.numpy()[:10])

Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Image batch shape: (64, 150, 150, 3)
Label batch shape: (64,)
Pixel range: 17.3332 to 255.0
First 10 labels: [3 7 5 2 9 6 7 1 0 1]


### **VGG16 with No Weights**
Importing the VGG16 model and for a baseline test we will be training it without freezing the weights, basically just using the architecture not the pre-trained values.

In [ ]:
from keras.applications.vgg16 import VGG16

In [ ]:
vgg_nw = VGG16(
    weights=None,
    include_top = False,
    input_shape=(150,150,3)
)

In [ ]:
model_vgg_nw = Sequential()

model_vgg_nw.add(vgg_nw)
model_vgg_nw.add(GlobalAveragePooling2D())
model_vgg_nw.add(Dense(256,activation='relu'))
model_vgg_nw.add(Dense(10,activation='softmax'))

In [ ]:
model_vgg_nw.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,848,586 (56.64 MB)

 Trainable params: 14,848,586 (56.64 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model_vgg_nw.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [ ]:
history_vgg_nw = model_vgg_nw.fit(train_ds,epochs=10,validation_data=validation_ds)

Epoch 1/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 267s 654ms/step - accuracy: 0.3259 - loss: 1.9975 - val_accuracy: 0.4896 - val_loss: 1.3686
Epoch 2/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 175s 517ms/step - accuracy: 0.6011 - loss: 1.0728 - val_accuracy: 0.7204 - val_loss: 0.7968
Epoch 3/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 174s 516ms/step - accuracy: 0.7015 - loss: 0.8220 - val_accuracy: 0.7604 - val_loss: 0.7044
Epoch 4/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 174s 515ms/step - accuracy: 0.7632 - loss: 0.6626 - val_accuracy: 0.7819 - val_loss: 0.6364
Epoch 5/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 174s 514ms/step - accuracy: 0.8007 - loss: 0.5688 - val_accuracy: 0.8057 - val_loss: 0.5737
Epoch 6/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 174s 514ms/step - accuracy: 0.8202 - loss: 0.5124 - val_accuracy: 0.8339 - val_loss: 0.4778
Epoch 7/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 174s 514ms/step - accuracy: 0.8438 - loss: 0.4407 - val_accuracy: 0.8037 - val_loss: 0.5519
Epoch 8/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 174s 514ms/step - accuracy: 0.8589 -

### **VGG16 with ImageNet Weights**
Now training it with freezing the weights from ImageNET dataset. We will just be training the Fully Connected layer without touching the Convolution blocks of VGG16. This will significantly decrease the training time as the parameters are reduced by half

In [ ]:
vgg_iw = VGG16(
    weights='imagenet',
    include_top = False,
    input_shape=(150,150,3)
)

In [ ]:
vgg_iw.trainable = False

In [ ]:
model_vgg_iw = Sequential()

model_vgg_iw.add(vgg_iw)
model_vgg_iw.add(Flatten())
model_vgg_iw.add(Dense(256,activation='relu'))
model_vgg_iw.add(Dense(10,activation='softmax'))

In [ ]:
model_vgg_iw.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,814,666 (64.14 MB)

 Trainable params: 2,099,978 (8.01 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [ ]:
model_vgg_iw.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [ ]:
history_vgg_iw = model_vgg_iw.fit(train_ds,epochs=10,validation_data=validation_ds)

Epoch 1/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 93s 268ms/step - accuracy: 0.8728 - loss: 0.6102 - val_accuracy: 0.9037 - val_loss: 0.3146
Epoch 2/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 81s 240ms/step - accuracy: 0.9592 - loss: 0.1241 - val_accuracy: 0.9204 - val_loss: 0.3008
Epoch 3/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 82s 240ms/step - accuracy: 0.9820 - loss: 0.0551 - val_accuracy: 0.9265 - val_loss: 0.3587
Epoch 4/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 81s 240ms/step - accuracy: 0.9839 - loss: 0.0540 - val_accuracy: 0.9159 - val_loss: 0.4008
Epoch 5/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 81s 240ms/step - accuracy: 0.9851 - loss: 0.0482 - val_accuracy: 0.9144 - val_loss: 0.4529
Epoch 6/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 81s 241ms/step - accuracy: 0.9854 - loss: 0.0474 - val_accuracy: 0.9230 - val_loss: 0.4779
Epoch 7/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 81s 241ms/step - accuracy: 0.9814 - loss: 0.0763 - val_accuracy: 0.9196 - val_loss: 0.6124
Epoch 8/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 81s 240ms/step - accuracy: 0.9818 - loss: 0

### **VGG16 Fine-Tuning**
Significantly better results in the previous training. Let's take it a step ahead by going another block deeper and de-freezing the Convolution block 5

In [ ]:
vgg_ft = VGG16(
    weights='imagenet',
    include_top = False,
    input_shape=(150,150,3)
)

In [ ]:
for layer in vgg_ft.layers:
    layer.trainable = False

for layer in vgg_ft.layers:
    if layer.name.startswith("block5"):
        layer.trainable = True

for layer in vgg_ft.layers:
    print(layer.name, layer.trainable)

input_layer_13 False
block1_conv1 False
block1_conv2 False
block1_pool False
block2_conv1 False
block2_conv2 False
block2_pool False
block3_conv1 False
block3_conv2 False
block3_conv3 False
block3_pool False
block4_conv1 False
block4_conv2 False
block4_conv3 False
block4_pool False
block5_conv1 True
block5_conv2 True
block5_conv3 True
block5_pool True


In [ ]:
model_vgg_ft = Sequential()

model_vgg_ft.add(vgg_ft)
model_vgg_ft.add(Flatten())
model_vgg_ft.add(Dense(256,activation='relu'))
model_vgg_ft.add(Dense(10,activation='softmax'))

In [ ]:
model_vgg_ft.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_6 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,814,666 (64.14 MB)

 Trainable params: 9,179,402 (35.02 MB)

 Non-trainable params: 7,635,264 (29.13 MB)

In [ ]:
model_vgg_ft.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_vgg_ft = model_vgg_ft.fit(train_ds,epochs=10,validation_data=validation_ds)

Epoch 1/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 100s 282ms/step - accuracy: 0.6678 - loss: 1.2551 - val_accuracy: 0.8719 - val_loss: 0.4456
Epoch 2/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 91s 269ms/step - accuracy: 0.9149 - loss: 0.2779 - val_accuracy: 0.9150 - val_loss: 0.3169
Epoch 3/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 91s 268ms/step - accuracy: 0.9625 - loss: 0.1291 - val_accuracy: 0.9235 - val_loss: 0.2822
Epoch 4/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 91s 269ms/step - accuracy: 0.9836 - loss: 0.0639 - val_accuracy: 0.9278 - val_loss: 0.2742
Epoch 5/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 91s 269ms/step - accuracy: 0.9935 - loss: 0.0331 - val_accuracy: 0.9311 - val_loss: 0.2718
Epoch 6/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 91s 269ms/step - accuracy: 0.9971 - loss: 0.0178 - val_accuracy: 0.9346 - val_loss: 0.2686
Epoch 7/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 91s 269ms/step - accuracy: 0.9991 - loss: 0.0098 - val_accuracy: 0.9346 - val_loss: 0.2719
Epoch 8/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 91s 269ms/step - accuracy: 0.9997 - loss: 

Amazing results, let's go another block deeper

### **VGG16 Fine Tuning V2**
Now de-freezing the 4th block too as an iteration

In [ ]:
vgg_ft_v2 = VGG16(
    weights='imagenet',
    include_top = False,
    input_shape=(150,150,3)
)

In [ ]:
for layer in vgg_ft_v2.layers:
    layer.trainable = False

for layer in vgg_ft_v2.layers:
    if layer.name.startswith(("block4", "block5")):
        layer.trainable = True

for layer in vgg_ft_v2.layers:
    print(layer.name, layer.trainable)

input_layer_16 False
block1_conv1 False
block1_conv2 False
block1_pool False
block2_conv1 False
block2_conv2 False
block2_pool False
block3_conv1 False
block3_conv2 False
block3_conv3 False
block3_pool False
block4_conv1 True
block4_conv2 True
block4_conv3 True
block4_pool True
block5_conv1 True
block5_conv2 True
block5_conv3 True
block5_pool True


In [ ]:
model_vgg_ft_v2 = Sequential()

model_vgg_ft_v2.add(vgg_ft_v2)
model_vgg_ft_v2.add(Flatten())
model_vgg_ft_v2.add(Dense(256,activation='relu'))
model_vgg_ft_v2.add(Dense(10,activation='softmax'))

In [ ]:
model_vgg_ft_v2.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_7 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,814,666 (64.14 MB)

 Trainable params: 15,079,178 (57.52 MB)

 Non-trainable params: 1,735,488 (6.62 MB)

In [ ]:
model_vgg_ft_v2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_vgg_ft_v2 = model_vgg_ft_v2.fit(train_ds,epochs=10,validation_data=validation_ds)

Epoch 1/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 122s 345ms/step - accuracy: 0.7174 - loss: 0.9791 - val_accuracy: 0.9026 - val_loss: 0.3075
Epoch 2/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 112s 330ms/step - accuracy: 0.9308 - loss: 0.2147 - val_accuracy: 0.9309 - val_loss: 0.2093
Epoch 3/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 112s 331ms/step - accuracy: 0.9650 - loss: 0.1069 - val_accuracy: 0.9483 - val_loss: 0.1700
Epoch 4/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 112s 331ms/step - accuracy: 0.9823 - loss: 0.0572 - val_accuracy: 0.9543 - val_loss: 0.1510
Epoch 5/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 112s 331ms/step - accuracy: 0.9910 - loss: 0.0323 - val_accuracy: 0.9504 - val_loss: 0.1665
Epoch 6/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 112s 331ms/step - accuracy: 0.9954 - loss: 0.0184 - val_accuracy: 0.9556 - val_loss: 0.1613
Epoch 7/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 112s 331ms/step - accuracy: 0.9976 - loss: 0.0110 - val_accuracy: 0.9522 - val_loss: 0.1796
Epoch 8/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 112s 331ms/step - accuracy: 0.9991 -

### **VGG16 Fine Tuning V2 with GAP**
Trying to reduce a bit of complexity with introducung Global Average Pooling as the Flatten Layer

In [ ]:
model_vgg_ft_v2_gap = Sequential()

model_vgg_ft_v2_gap.add(vgg_ft_v2)
model_vgg_ft_v2_gap.add(GlobalAveragePooling2D())
model_vgg_ft_v2_gap.add(Dense(256,activation='relu'))
model_vgg_ft_v2_gap.add(Dense(10,activation='softmax'))

In [ ]:
model_vgg_ft_v2_gap.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,848,586 (56.64 MB)

 Trainable params: 13,113,098 (50.02 MB)

 Non-trainable params: 1,735,488 (6.62 MB)

In [ ]:
model_vgg_ft_v2_gap.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_vgg_ft_v2_gap = model_vgg_ft_v2_gap.fit(train_ds,epochs=10,validation_data=validation_ds)

Epoch 1/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 121s 343ms/step - accuracy: 0.8910 - loss: 0.3876 - val_accuracy: 0.9622 - val_loss: 0.1288
Epoch 2/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 116s 344ms/step - accuracy: 0.9760 - loss: 0.0777 - val_accuracy: 0.9620 - val_loss: 0.1250
Epoch 3/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 112s 331ms/step - accuracy: 0.9864 - loss: 0.0431 - val_accuracy: 0.9659 - val_loss: 0.1182
Epoch 4/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 142s 332ms/step - accuracy: 0.9923 - loss: 0.0249 - val_accuracy: 0.9663 - val_loss: 0.1205
Epoch 5/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 112s 331ms/step - accuracy: 0.9943 - loss: 0.0184 - val_accuracy: 0.9669 - val_loss: 0.1292
Epoch 6/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 112s 330ms/step - accuracy: 0.9973 - loss: 0.0099 - val_accuracy: 0.9676 - val_loss: 0.1282
Epoch 7/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 112s 331ms/step - accuracy: 0.9950 - loss: 0.0157 - val_accuracy: 0.9676 - val_loss: 0.1237
Epoch 8/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 112s 331ms/step - accuracy: 0.9963 -

It did help reduce bit of overfitting but complexity largely remains same for training

### **Introducing Dropout**
Experimenting by dropout layers to see if theres any improvement in generalization

In [ ]:
model_vgg_ft_v2_gap_dropout = Sequential()

model_vgg_ft_v2_gap_dropout.add(vgg_ft_v2)
model_vgg_ft_v2_gap_dropout.add(GlobalAveragePooling2D())
model_vgg_ft_v2_gap_dropout.add(Dense(256,activation='relu'))
model_vgg_ft_v2_gap_dropout.add(Dropout(0.5))
model_vgg_ft_v2_gap_dropout.add(Dense(10,activation='softmax'))

In [ ]:
model_vgg_ft_v2_gap_dropout.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,848,586 (56.64 MB)

 Trainable params: 13,113,098 (50.02 MB)

 Non-trainable params: 1,735,488 (6.62 MB)

In [ ]:
model_vgg_ft_v2_gap_dropout.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_vgg_ft_v2_gap_dropout = model_vgg_ft_v2_gap_dropout.fit(train_ds,epochs=10,validation_data=validation_ds)

Epoch 1/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 123s 347ms/step - accuracy: 0.7450 - loss: 1.0183 - val_accuracy: 0.9631 - val_loss: 0.1205
Epoch 2/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 134s 331ms/step - accuracy: 0.9556 - loss: 0.1519 - val_accuracy: 0.9720 - val_loss: 0.0951
Epoch 3/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 112s 330ms/step - accuracy: 0.9733 - loss: 0.0924 - val_accuracy: 0.9726 - val_loss: 0.0940
Epoch 4/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 111s 330ms/step - accuracy: 0.9808 - loss: 0.0644 - val_accuracy: 0.9746 - val_loss: 0.0933
Epoch 5/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 147s 344ms/step - accuracy: 0.9854 - loss: 0.0468 - val_accuracy: 0.9693 - val_loss: 0.1182
Epoch 6/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 112s 330ms/step - accuracy: 0.9869 - loss: 0.0408 - val_accuracy: 0.9774 - val_loss: 0.0921
Epoch 7/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 112s 330ms/step - accuracy: 0.9899 - loss: 0.0314 - val_accuracy: 0.9715 - val_loss: 0.1195
Epoch 8/10
338/338 ━━━━━━━━━━━━━━━━━━━━ 112s 330ms/step - accuracy: 0.9919 -

Looks like we have reached a saturation point

###**Saving the Model**

In [ ]:
from tensorflow.keras.models import load_model
from google.colab import files

# Save model_v4 before downloading
model_vgg_ft_v2_gap_dropout.save("EuroSAT_VGG16.keras")
files.download("EuroSAT_VGG16.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

history_df = pd.DataFrame(history_vgg_ft_v2_gap_dropout.history)
history_df.to_csv('EuroSAT.csv', index=False)
files.download('EuroSAT.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>